In [2]:
STMT_KEYWORDS = {
    "select", "insert", "update", "delete", "merge", "drop", "create",
    "alter", "truncate", "declare", "set", "with", "while", "if", "begin",
    "print", "waitfor", "grant", "revoke", "use", "open", "close", "fetch",
    "return", "goto", "throw", "commit", "rollback", "save", "backup",
    "restore", "bulk",
}

def _strip(sql):
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.DOTALL)   # block comments
    sql = re.sub(r"--[^\n]*", " ", sql)                      # line comments
    sql = re.sub(r"'(?:[^']|'')*'", " ", sql)                # string literals
    sql = re.sub(r"\[[^\]]*\]", " ", sql)                    # [bracketed] names
    return sql

def is_pure_exec(query):
    """True only if the query is one or more exec statements and nothing else."""
    s = _strip(query or "")
    if not re.match(r"^\s*exec(ute)?\b", s, re.IGNORECASE):
        return False
    for tok in re.findall(r"(?<![@#$\w])([A-Za-z_]+)", s):
        if tok.lower() in STMT_KEYWORDS:
            return False
    return True

In [3]:
import json
import re
import zipfile
from pathlib import Path

import pandas as pd
# Matches queries that start with EXEC or EXECUTE (case-insensitive),
# ignoring leading whitespace. \b prevents matching words like "executive".
EXEC_RE = re.compile(r"^\s*exec(ute)?\b", re.IGNORECASE)


# 1 Updated classifier

In [5]:
from classify_unparsable import classify_zip
def classify_zip(zip_path):
    rows = []
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(n for n in zf.namelist()
                       if n.lower().endswith(".json") and not n.endswith("/"))
        for name in names:
            row = {"file": Path(name).name}
            try:
                data = json.loads(zf.read(name).decode("utf-8"))
                texts = [q.get("query", "") for q in data.get("unparsable_queries", [])]
                impure = [t for t in texts if not is_pure_exec(t)]
                if not texts:
                    verdict = "No unparsable queries"
                elif impure:
                    verdict = "Needs manual review"
                else:
                    verdict = "No manual review"
                row.update(verdict=verdict, num_queries=len(texts),
                           num_pure_exec=len(texts) - len(impure), num_review=len(impure))
            except Exception as e:
                row.update(verdict="Parse error", num_queries=None,
                           num_pure_exec=None, num_review=None)
                row["error"] = f"{type(e).__name__}: {e}"
            rows.append(row)
    return pd.DataFrame(rows)

def extract_review_queries(zip_path):
    rows = []
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(n for n in zf.namelist()
                       if n.lower().endswith(".json") and not n.endswith("/"))
        for name in names:
            try:
                data = json.loads(zf.read(name).decode("utf-8"))
            except Exception:
                continue
            for i, qd in enumerate(data.get("unparsable_queries", [])):
                text = qd.get("query", "")
                if not is_pure_exec(text):
                    rows.append({
                        "file": Path(name).name,
                        "query_index": i,
                        "query": text,
                        "error": qd.get("error", ""),
                    })
    return pd.DataFrame(rows, columns=["file", "query_index", "query", "error"])

# print_report(df)   # still works; rename num_non_exec -> num_review in it if you kept that

queries_df = extract_review_queries("unparsable-queries (5).zip")
display(queries_df)
queries_df.to_excel("review_queries_report_11062026_3.xlsx", index=False)

,file,query_index,query,error
0,unparsable_queries_DATA_CONTROL-DQ-proc_load_d...,0,update dc set sql_statement = sql_statement + ...,"Invalid expression / Unexpected token. Line 1,..."
1,unparsable_queries_DM_NWB-BRF-proc_report_rent...,1,update tb set ind_max_lening = case when fl.ma...,"Invalid expression / Unexpected token. Line 1,..."
2,unparsable_queries_DM_NWB-BRF-proc_report_sald...,0,"select fs.interest_rate , fs.spread_rate , ds....","Invalid expression / Unexpected token. Line 1,..."
3,unparsable_queries_DM_NWB-BRF-proc_send_email_...,0,"throw 50000, 'Invalid e-mail', 1","Invalid expression / Unexpected token. Line 1,..."
4,unparsable_queries_DM_NWB-CPL-proc_load_letter...,1,waitfor delay '00:01',"Invalid expression / Unexpected token. Line 1,..."
...,...,...,...,...
272,unparsable_queries_MFVHA-HA-proc_load_mfvha_ef...,0,drop table if exists [#csa_calculation_ret] se...,"Invalid expression / Unexpected token. Line 1,..."
273,unparsable_queries_MFVHA-HA-proc_load_valuatio...,0,"throw 50000, @msg_message, 1","Invalid expression / Unexpected token. Line 1,..."
274,unparsable_queries_MFVHA-HA-proc_match_designa...,0,"throw 50000, @msg_message, 1","Invalid expression / Unexpected token. Line 1,..."
275,unparsable_queries_MFVHA-HA-proc_run_mfvha_que...,0,"throw 50000, @msg_message, 1","Invalid expression / Unexpected token. Line 1,..."


IllegalCharacterError: Invalid expression / Unexpected token. Line 1, Col: 72.
  update dc set sql_statement = sql_statement + djs.join_statementfrom [4mpit[0m.dq_control as dc inner join ( select djs.dq_control_id , join_statement = string_agg(djs.join_state cannot be used in worksheets.